# Advanced AML Name Screening & Workload Optimization Platform
This notebook implements:
- Feature Engineering
- Fuzzy Name Matching
- ML Risk Model (Logistic Regression)
- Analyst Workload Optimization
- Schema Adaptive Column Detection
- Unified Risk Scoring


In [ ]:
!pip install rapidfuzz scikit-learn pandas numpy joblib

In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

## Feature Engineering Functions

In [ ]:
def normalize(text):
    return text.lower().strip()

def name_similarity(a, b):
    return fuzz.token_sort_ratio(a, b) / 100.0

def dob_similarity(d1, d2):
    return 1.0 if d1 == d2 else 0.0

def country_match(c1, c2):
    return 1.0 if c1 == c2 else 0.0

def gender_match(g1, g2):
    return 1.0 if g1 == g2 else 0.0

def generate_features(customer, watchlist):
    return [
        name_similarity(normalize(customer["name"]), normalize(watchlist["name"])),
        dob_similarity(customer["dob"], watchlist["dob"]),
        country_match(customer["country"], watchlist["country"]),
        gender_match(customer["gender"], watchlist["gender"]),
        watchlist["severity"]
    ]

## Simulated Training Dataset

In [ ]:
X = np.array([
    [0.95,1,1,1,0.9],
    [0.90,0,0,1,0.9],
    [0.85,1,0,1,0.8],
    [0.40,0,0,0,0.3],
    [0.99,1,1,1,1.0],
    [0.70,0,0,0,0.6]
])
y = np.array([1,0,1,0,1,0])

model = LogisticRegression()
model.fit(X,y)

joblib.dump(model,"risk_model.pkl")

print("Model trained successfully")

## Prediction & Decision Logic

In [ ]:
def predict_risk(features):
    return model.predict_proba([features])[0][1]

def decision_logic(prob):
    if prob > 0.8:
        return "HIGH PRIORITY REVIEW"
    elif prob > 0.4:
        return "MEDIUM PRIORITY REVIEW"
    else:
        return "LOW PRIORITY / AUTO CLOSE CANDIDATE" 

## Test Example

In [ ]:
customer = {
    "name":"Mohammad Ali",
    "dob":"1995-04-10",
    "country":"India",
    "gender":"Male"
}

watchlist = {
    "name":"Muhammad Ali",
    "dob":"1960-01-01",
    "country":"Syria",
    "gender":"Male",
    "severity":0.9
}

features = generate_features(customer, watchlist)
prob = predict_risk(features)
decision = decision_logic(prob)

print("Features:", features)
print("True Match Probability:", prob)
print("Decision:", decision)

## Schema Adaptive Column Detection

In [ ]:
ONTOLOGY = {
    "amount": ["amount","txn_amt","value"],
    "country": ["country","ctry"],
    "name": ["name","customer_name"]
}

def detect_columns(df):
    mapping = {}
    for col in df.columns:
        for key, candidates in ONTOLOGY.items():
            for candidate in candidates:
                if fuzz.partial_ratio(col.lower(), candidate.lower()) > 80:
                    mapping[key] = col
    return mapping

In [ ]:
# Example CSV simulation
data = pd.DataFrame({
    "cust_name":["Ali","Rahul"],
    "txn_amt":[120000,50000],
    "country":["Iran","India"]
})

mapping = detect_columns(data)
print("Detected Mapping:", mapping)

## Unified Risk Scoring

In [ ]:
def unified_risk(df, mapping):
    risk_scores = []
    
    if "amount" in mapping:
        avg_amt = df[mapping["amount"]].mean()
        risk_scores.append(0.9 if avg_amt > 100000 else 0.3)
        
    if "country" in mapping:
        high_risk = ["Iran","North Korea","Syria"]
        risk_scores.append(0.8 if df[mapping["country"]].isin(high_risk).any() else 0.2)
        
    return np.mean(risk_scores)

print("Unified Dataset Risk:", unified_risk(data, mapping))